# Branches Exploratory Data Analysis

## Purpose

This notebook checks whether branch records are complete, consistent and suitable for joining to organisations, programmes and downstream Pulse80 application data.

## Files used

- `Branches.csv` — branch definitions
- `Organisations.csv` — parent organisation definitions
- `Programmes.csv` — programmes assigned to organisations and branches

The current sample contains two branches belonging to one organisation. The checks confirm the current records, but a larger dataset will still be needed to judge behaviour across many organisations.

## 1. Prepare the notebook

The following block imports pandas and locates the repository's raw-data folder. It uses a relative search so the notebook works on different computers and does not contain or print anyone's personal file path.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (
            directory / "data" / "raw",
            directory / "data-analytics" / "data" / "raw",
        ):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate the repository raw-data folder.")

RAW_DATA_DIR = find_raw_data_dir()

## 2. Load the related datasets

This block loads the branch, organisation and programme files. The summary confirms how many rows and columns are available before any analysis is performed.

In [ ]:
branches = pd.read_csv(RAW_DATA_DIR / "Branches.csv")
organisations = pd.read_csv(RAW_DATA_DIR / "Organisations.csv")
programmes = pd.read_csv(RAW_DATA_DIR / "Programmes.csv")

dataset_summary = pd.DataFrame({
    "dataset": ["Branches", "Organisations", "Programmes"],
    "rows": [len(branches), len(organisations), len(programmes)],
    "columns": [
        len(branches.columns),
        len(organisations.columns),
        len(programmes.columns),
    ],
})

dataset_summary

## 3. Inspect the branch records

This block displays the branch records so their identifiers, names, cities, countries and creation dates can be reviewed directly.

In [ ]:
branches

## 4. Profile completeness and uniqueness

This block summarises each column's data type, missing values and number of unique values. It also checks whether any complete rows have been duplicated.

In [ ]:
branch_profile = pd.DataFrame({
    "data_type": branches.dtypes.astype(str),
    "missing_count": branches.isna().sum(),
    "unique_values": branches.nunique(dropna=False),
})

print("Fully duplicated rows:", branches.duplicated().sum())
branch_profile

## 5. Validate branch identifiers and names

This block checks that every branch has an ID, each ID is unique, IDs follow the `BR-001` format, and the same organisation does not contain duplicate branch names after ignoring capitalisation and extra spaces.

In [ ]:
normalised_branch_names = branches["name"].astype("string").str.strip().str.casefold()

identifier_checks = pd.Series({
    "missing_branch_ids": branches["branch_id"].isna().sum(),
    "duplicate_branch_ids": branches["branch_id"].duplicated().sum(),
    "invalid_branch_id_formats": (
        ~branches["branch_id"].astype("string").str.match(r"^BR-[0-9]{3,}$", na=False)
    ).sum(),
    "duplicate_names_within_organisation": (
        branches.assign(_normalised_name=normalised_branch_names)
        .duplicated(subset=["organisation_id", "_normalised_name"])
        .sum()
    ),
})

identifier_checks.to_frame("count")

### Confirm the identifier rules

These assertions stop the notebook if a required identifier rule fails. Passing them means branch IDs are present, unique, consistently formatted and safe to use as join keys.

In [ ]:
assert identifier_checks.eq(0).all()
print("Branch identifier and name checks passed.")

## 6. Validate required text and creation dates

This block checks for blank branch names, cities, countries and organisation IDs. It also converts `created_at` into a real timestamp and detects invalid dates.

In [ ]:
required_text_columns = ["organisation_id", "name", "city", "country"]

blank_text_counts = (
    branches[required_text_columns]
    .astype("string")
    .apply(lambda column: column.str.strip().eq("").sum())
)

branches["created_at"] = pd.to_datetime(
    branches["created_at"],
    utc=True,
    errors="coerce",
)

value_checks = pd.concat([
    blank_text_counts.rename(lambda name: f"blank_{name}"),
    pd.Series({"invalid_created_at_dates": branches["created_at"].isna().sum()}),
])

value_checks.to_frame("count")

### Confirm the required values

These assertions verify that required branch details are not blank and creation dates are readable. Passing them means the current records contain the basic information the application needs.

In [ ]:
assert value_checks.eq(0).all()
print("Required branch value checks passed.")

## 7. Validate the organisation relationship

Each branch must belong to an organisation that exists. This block joins branches to organisations, checks that no organisation reference is missing, and compares the branch country with the parent organisation's country.

In [ ]:
branch_organisation_join = branches.merge(
    organisations[["organisation_id", "name", "country"]].rename(
        columns={
            "name": "organisation_name",
            "country": "organisation_country",
        }
    ),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
)

branch_organisation_join["country_matches"] = (
    branch_organisation_join["country"].astype("string").str.strip().str.casefold()
    == branch_organisation_join["organisation_country"].astype("string").str.strip().str.casefold()
)

branch_organisation_join

### Confirm the organisation join

These assertions confirm that every branch found its parent organisation, the countries agree, and the join did not add or remove branch rows.

In [ ]:
assert branch_organisation_join["organisation_join_status"].eq("both").all()
assert branch_organisation_join["country_matches"].all()
assert len(branch_organisation_join) == len(branches)
print("All branches join to their organisations correctly.")

## 8. Validate programme use of branches

Programmes refer to branches through `branch_id`. This block checks that every programme points to an existing branch and that the selected branch belongs to the same organisation as the programme.

In [ ]:
programme_branch_join = programmes.merge(
    branches[["branch_id", "organisation_id", "name"]].rename(
        columns={
            "organisation_id": "branch_organisation_id",
            "name": "branch_name",
        }
    ),
    on="branch_id",
    how="left",
    validate="many_to_one",
    indicator="branch_join_status",
)

programme_branch_join["branch_belongs_to_programme_organisation"] = (
    programme_branch_join["organisation_id"]
    == programme_branch_join["branch_organisation_id"]
)

programme_branch_join

### Confirm the programme join

These assertions confirm that programme branch references exist, respect organisation ownership and preserve every programme row.

In [ ]:
assert programme_branch_join["branch_join_status"].eq("both").all()
assert programme_branch_join["branch_belongs_to_programme_organisation"].all()
assert len(programme_branch_join) == len(programmes)
print("All programmes join to the correct branches.")

## 9. Measure programme coverage by branch

This block counts programmes assigned to each branch. A branch with zero programmes is not necessarily an error; it may simply be available for future programme delivery.

In [ ]:
programmes_per_branch = (
    programmes.groupby("branch_id")
    .size()
    .rename("programme_count")
)

branch_programme_coverage = (
    branches[["branch_id", "organisation_id", "name", "city", "country"]]
    .merge(
        programmes_per_branch,
        left_on="branch_id",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)

branch_programme_coverage["programme_count"] = (
    branch_programme_coverage["programme_count"].fillna(0).astype(int)
)

branch_programme_coverage

## 10. Summarise join readiness

This block brings the important validation results into one table. A value of zero for every failed check means the current branch data is suitable for downstream joins.

In [ ]:
join_readiness = pd.Series({
    "missing_branch_ids": branches["branch_id"].isna().sum(),
    "duplicate_branch_ids": branches["branch_id"].duplicated().sum(),
    "invalid_branch_id_formats": identifier_checks["invalid_branch_id_formats"],
    "invalid_organisation_references": (
        ~branch_organisation_join["organisation_join_status"].eq("both")
    ).sum(),
    "branch_organisation_country_mismatches": (
        ~branch_organisation_join["country_matches"]
    ).sum(),
    "invalid_programme_branch_references": (
        ~programme_branch_join["branch_join_status"].eq("both")
    ).sum(),
    "programme_branch_organisation_mismatches": (
        ~programme_branch_join["branch_belongs_to_programme_organisation"]
    ).sum(),
})

join_readiness.to_frame("failed_records")

### Final automated confirmation

This final assertion provides one clear pass or fail result for the current dataset. It fails immediately if any critical identifier or relationship problem remains.

In [ ]:
assert join_readiness.eq(0).all()
print("All critical branch join-readiness checks passed.")

## 11. Simple conclusion

The current branch data works correctly.

- Both branch records have complete names, cities, countries and creation dates.
- Each branch has a unique identifier written in the expected format.
- Both branches belong to an organisation that exists.
- The branch countries agree with the parent organisation's country.
- Every programme points to a branch that exists.
- Each programme's branch belongs to the same organisation as the programme.
- No records are lost when the datasets are joined.

One branch currently has the wellness programme, while the other branch has no programme assigned. This does not mean the second branch is wrong. It means that the branch exists and can be used for a future programme.

There is one important limitation: the file contains only two branches belonging to one organisation. The current records pass the checks, but more data is needed to confirm that names, cities and countries remain consistent when Pulse80 supports many organisations and locations.

**Overall result:** `Branches.csv` is complete and suitable for joining to organisations, programmes and downstream application data.